# Chuẩn bị dữ liệu tóm tắt tiếng Việt



## 1. Kết nối và môi trường


In [1]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive/vietnamese-summarization")
drive_root.mkdir(parents=True, exist_ok=True)

print("Thư mục dự án trên Drive:", drive_root)


Mounted at /content/drive
Thư mục dự án trên Drive: /content/drive/MyDrive/vietnamese-summarization


In [2]:
import os
import sys
import subprocess

repo_url = "https://github.com/MD1809/vietnamese-summarization.git"
repo_dir = Path("/content/vietnamese-summarization")
target_commit = "b21dc1b3313021c88ff37a834fda5b3cdb83b2f3"

if not repo_dir.exists():
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)

remote_result = subprocess.run(
    ["git", "remote", "get-url", "origin"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if remote_result.stdout.strip() != repo_url:
    raise RuntimeError("Lỗi: Repository không đúng dự án.")

status_result = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if status_result.stdout.strip() != "":
    raise RuntimeError("Lỗi: Repository còn thay đổi. Hãy dùng runtime mới.")

subprocess.run(["git", "fetch", "origin"], cwd=repo_dir, check=True)
subprocess.run(
    ["git", "checkout", "--detach", target_commit],
    cwd=repo_dir,
    check=True,
)

os.chdir(repo_dir)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("Commit sử dụng:", target_commit)


Commit sử dụng: b21dc1b3313021c88ff37a834fda5b3cdb83b2f3


In [3]:
%pip uninstall -y -q diffusers torchvision timm fastai
%pip install -q "jedi==0.20.1"
%pip install -q -r /content/vietnamese-summarization/requirements.txt
%pip check


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
No broken requirements found.


In [4]:
import json
import src.data
import src.preprocessing
from src.utils import load_train_config

config_path = repo_dir / "configs" / "train.json"
train_config = load_train_config(config_path)

manifest_names = [
    "wikilingua_source.json",
    "wikilingua_cleaning.json",
    "wikilingua_splits.json",
    "wikilingua_subsets.json",
    "wikilingua_statistics.json",
    "wikilingua_processed.json",
    "wikilingua_truncation_1024_128.json",
]

reference_manifests = {}

for name in manifest_names:
    manifest_path = repo_dir / "data" / "manifests" / name
    reference_manifests[name] = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )

print(json.dumps(
    {
        "dataset": train_config["dataset"],
        "seed": train_config["seed"],
        "max_source_length": train_config["max_source_length"],
        "max_target_length": train_config["max_target_length"],
    },
    ensure_ascii=False,
    indent=2,
))


{
  "dataset": {
    "id": "esdurmus/wiki_lingua",
    "revision": "ea3db3510cbd34d0f8dc612419ae40e4732f3b40",
    "language": "vietnamese"
  },
  "seed": 42,
  "max_source_length": 1024,
  "max_target_length": 128
}


## 2. Tải nguồn và chuyển section thành cặp


In [5]:
cache_dir = drive_root / "data" / "raw"
dataset = src.data.load_wikilingua(cache_dir, config_path)
source_rows = dataset["train"]

expected_article_count = reference_manifests[
    "wikilingua_source.json"
]["splits"]["train"]["article_row_count"]

if len(source_rows) != expected_article_count:
    raise RuntimeError("Lỗi: Số bài nguồn khác manifest đã chốt.")

pairs = src.data.convert_wikilingua_sections(source_rows)

print("Số bài nguồn:", len(source_rows))
print("Số cặp trước làm sạch:", len(pairs))


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Số bài nguồn: 6616
Số cặp trước làm sạch: 19581


## 3. Chuẩn hóa và làm sạch


In [6]:
normalized_pairs = src.preprocessing.normalize_pairs(pairs)

cleaned_pairs, cleaning_report = src.data.clean_wikilingua_pairs(
    normalized_pairs
)

expected_cleaning = reference_manifests["wikilingua_cleaning.json"].copy()
expected_cleaning.pop("created_at_utc", None)
expected_cleaning.pop("dataset", None)

if cleaning_report != expected_cleaning:
    raise RuntimeError("Lỗi: Kết quả làm sạch khác dữ liệu đã chốt.")

print("Số cặp giữ lại:", len(cleaned_pairs))
print("Số cặp bị loại:", cleaning_report["removed_pair_count"])
print("Lý do loại:", cleaning_report["removed_counts_by_reason"])


Số cặp giữ lại: 19577
Số cặp bị loại: 4
Lý do loại: {'exact_duplicate_pair': 3, 'empty_document': 1}


## 4. Chia nhóm và khôi phục tập con cố định

In [7]:
dataset_splits, split_manifest = src.data.split_wikilingua_pairs(
    cleaned_pairs,
    cleaning_report,
    config_path,
)

expected_splits = reference_manifests["wikilingua_splits.json"].copy()
expected_splits.pop("created_at_utc", None)

if split_manifest != expected_splits:
    raise RuntimeError("Lỗi: Bộ chia khác manifest đã chốt.")

subset_manifest = reference_manifests["wikilingua_subsets.json"]

subset_splits = src.data.restore_fixed_subsets(
    dataset_splits,
    split_manifest,
    subset_manifest,
)

split_counts = {}

for split_name in ["train", "validation", "test"]:
    split_counts[split_name] = len(subset_splits[split_name])

print(json.dumps(split_counts, ensure_ascii=False, indent=2))


{
  "train": 2000,
  "validation": 201,
  "test": 200
}


## 5. Thống kê cần thiết

In [8]:
dataset_statistics = src.data.build_dataset_statistics(
    subset_splits,
    subset_manifest,
    cleaning_report,
    config_path,
)

if dataset_statistics != reference_manifests["wikilingua_statistics.json"]:
    raise RuntimeError("Lỗi: Thống kê khác kết quả đã chốt.")

print(json.dumps(
    {
        "splits": dataset_statistics["splits"],
        "training_token_lengths": dataset_statistics["training_token_lengths"],
    },
    ensure_ascii=False,
    indent=2,
))


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


{
  "splits": {
    "train": {
      "pair_count": 2000,
      "article_count": 679
    },
    "validation": {
      "pair_count": 201,
      "article_count": 65
    },
    "test": {
      "pair_count": 200,
      "article_count": 65
    }
  },
  "training_token_lengths": {
    "sample_count": 2000,
    "document": {
      "mean": 595.649,
      "p95": 1315,
      "max": 10351
    },
    "summary": {
      "mean": 61.564,
      "p95": 128,
      "max": 711
    }
  }
}


## 6. Xuất JSONL và manifest

In [9]:
drive_manifest_dir = drive_root / "data" / "manifests"

saved_manifest_names = [
    "wikilingua_source.json",
    "wikilingua_cleaning.json",
    "wikilingua_splits.json",
    "wikilingua_subsets.json",
    "wikilingua_truncation_1024_128.json",
]

for name in saved_manifest_names:
    target_path = drive_manifest_dir / name

    if target_path.exists():
        current_manifest = json.loads(target_path.read_text(encoding="utf-8"))
        expected_manifest = reference_manifests[name].copy()
        current_manifest.pop("created_at_utc", None)
        expected_manifest.pop("created_at_utc", None)

        if current_manifest != expected_manifest:
            raise RuntimeError("Lỗi: Manifest trên Drive không khớp: " + name)

processed_manifest = src.data.export_prepared_datasets(
    subset_splits,
    subset_manifest,
    dataset_statistics,
    drive_root,
    config_path,
)

actual_processed = processed_manifest.copy()
expected_processed = reference_manifests["wikilingua_processed.json"].copy()
actual_processed.pop("created_at_utc", None)
expected_processed.pop("created_at_utc", None)

if actual_processed != expected_processed:
    raise RuntimeError("Lỗi: Dữ liệu xuất khác bộ dữ liệu đã chốt.")

drive_manifest_dir.mkdir(parents=True, exist_ok=True)

for name in saved_manifest_names:
    target_path = drive_manifest_dir / name

    if not target_path.exists():
        source_path = repo_dir / "data" / "manifests" / name
        target_path.write_bytes(source_path.read_bytes())

print("Đã hoàn tất chuẩn bị dữ liệu.")
print("Dữ liệu:", drive_root / "data" / "processed")
print("Manifest:", drive_manifest_dir)
print("Số mẫu:", split_counts)
print("Khóa test:", processed_manifest["test_lock"]["locked"])


Đã hoàn tất chuẩn bị dữ liệu.
Dữ liệu: /content/drive/MyDrive/vietnamese-summarization/data/processed
Manifest: /content/drive/MyDrive/vietnamese-summarization/data/manifests
Số mẫu: {'train': 2000, 'validation': 201, 'test': 200}
Khóa test: True
